In [ ]:
#@title <center><h2><b>🚀 Leech-TJ • Heroku Deploy (Colab) ⚡</b></h2></center><br>

#@markdown <center><img src='https://raw.githubusercontent.com/Tj-Bots/Leech-TJ/wzgram/docs/banner-tj.png' width="640" alt="Leech-TJ"/></center>

#@markdown ---

#@markdown 📦 **Repo:** https://github.com/Tj-Bots/Leech-TJ <br>
#@markdown 🌿 **Branch:** wzgram <br>
#@markdown ⚙️ **Repo Version:** v1.4.0-x <br>
#@markdown ☢️ **Colab Version:** v2.0 <br>
#@markdown 📢 **Updates Channel:** https://t.me/TJ_Bots <br>
#@markdown 💬 **Support Group:** https://t.me/Tj_Leech_Group <br>

#@markdown ---

#@markdown ▶️ **Run the cells from top to bottom: 1 → 2 → 3 → 4 → 5.** <br>
#@markdown 🔒 Your settings go to Heroku **Config Vars**, never into git. <br>
#@markdown ⚠️ Do not use *Save a copy* after filling the form, it saves your secrets. <br>
#@markdown ⚠️ Heroku can suspend accounts that run bots, a VPS is safer.


In [ ]:
#@title <center><h3>🔐 <b>1 · Heroku Login</b></h3></center>
Heroku_Email = ""  #@param {type:"string"}
Heroku_API_Key = ""  #@param {type:"string"}
#@markdown <h6><i>🔑 API key: dashboard.heroku.com/account → API Key → Reveal</i></h6>

import os, re, shutil, subprocess

def run(cmd, check=True, show=True, cwd=None):
    """Runs a command (a list, no shell) and returns its output."""
    res = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    out = (res.stdout or "") + (res.stderr or "")
    if show and out.strip():
        print(out.strip())
    if check and res.returncode != 0:
        raise RuntimeError(f"Failed: {' '.join(cmd[:3])} ...")
    return out

if not Heroku_Email.strip() or not Heroku_API_Key.strip():
    raise ValueError("⚠️ Fill Heroku_Email and Heroku_API_Key")

if shutil.which("heroku") is None:
    print("📥 Installing the Heroku CLI...")
    subprocess.run("curl -s https://cli-assets.heroku.com/install.sh | sh", shell=True, check=True)

netrc = os.path.expanduser("~/.netrc")
with open(netrc, "w") as f:
    for host in ("api.heroku.com", "git.heroku.com"):
        f.write(f"machine {host}\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\n")
os.chmod(netrc, 0o600)

print("✅ Logged in as:", run(["heroku", "auth:whoami"], show=False).strip())


In [ ]:
#@title <center><h3>⚡ <b>2 · Create the Heroku App</b></h3></center>
App_Name = ""  #@param {type:"string"}
Server_Region = "eu"  #@param ["us", "eu"]
Team_Name = ""  #@param {type:"string"}
#@markdown <h6><i>🌍 Region by Telegram DC: DC2 / DC4 (Amsterdam) → eu (Dublin) · DC1 / DC3 (Miami) → us (Virginia) · DC5 (Singapore) → eu. Your DC is in the bot's /stats or @userinfobot</i></h6>
#@markdown <h6><i>📛 App name: lowercase letters, digits and -, 3 to 30 characters</i></h6>
#@markdown <h6><i>👥 Team_Name: only if the app belongs to a Heroku Team, leave empty for your personal account</i></h6>

if not re.fullmatch(r"[a-z][a-z0-9-]{2,29}", App_Name.strip()):
    raise ValueError("⚠️ App_Name: 3-30 chars, lowercase letters, digits and -, starting with a letter")
App_Name = App_Name.strip()

team = ["--team", Team_Name.strip()] if Team_Name.strip() else []
exists = subprocess.run(["heroku", "apps:info", "-a", App_Name], capture_output=True).returncode == 0
if exists:
    print(f"ℹ️ The app {App_Name} already exists, switching it to the container stack")
    run(["heroku", "stack:set", "container", "-a", App_Name])
else:
    run(["heroku", "create", App_Name, "--region", Server_Region, "--stack", "container", *team])
print("✅ App ready:", f"https://{App_Name}.herokuapp.com")


In [ ]:
#@title <center><h3>⚙️ <b>3 · Bot Settings</b></h3></center>
#@markdown #### 📝 ***Fill these (all required)***
BOT_TOKEN = ""  #@param {type:"string"}
TELEGRAM_API = ""  #@param {type:"string"}
TELEGRAM_HASH = ""  #@param {type:"string"}
OWNER_ID = ""  #@param {type:"string"}
DATABASE_URL = ""  #@param {type:"string"}
#@markdown <h6><i>🗄️ DATABASE_URL: MongoDB (mongodb+srv://...), needed so your settings survive restarts</i></h6>

#@markdown ---
#@markdown #### 📦 ***Or use your own `config.env`*** (optional, instead of the fields above)
Upload_Config = False  #@param {type:"boolean"}
#@markdown <h6><i>📤 Tick to upload your config.env from your computer</i></h6>
Config_File_URL = ""  #@param {type:"string"}
#@markdown <h6><i>🌐 Or a private Gist link to it (gist.github.com → Raw)</i></h6>

#@markdown ---
#@markdown #### ➕ ***Extra settings*** (optional)
Extra_Config = ""  #@param {type:"string"}
#@markdown <h6><i>🧩 `KEY=VALUE` separated by `;` , example: `RICH_MESSAGES=True;PM_ACCESS=True`</i></h6>
Upstream_Repo = "https://github.com/Tj-Bots/Leech-TJ"  #@param {type:"string"}
Upstream_Branch = "wzgram"  #@param {type:"string"}

def parse_env(text):
    """KEY = "VALUE" lines of a config.env -> dict (comments and empty values skipped)."""
    result = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        value = value.strip()
        if value[:1] in ("'", '"'):
            end = value.find(value[0], 1)
            if end == -1:
                continue  # a value over several lines, set it in Extra_Config instead
            value = value[1:end]
        else:
            value = re.sub(r"\s+#.*$", "", value)
        if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", key.strip()) and value != "":
            result[key.strip()] = value
    return result

config = {}
if Upload_Config:
    from google.colab import files
    config.update(parse_env(list(files.upload().values())[0].decode()))
elif Config_File_URL.strip():
    import urllib.request
    with urllib.request.urlopen(Config_File_URL.strip(), timeout=30) as resp:
        config.update(parse_env(resp.read().decode()))
else:
    config.update({
        "BOT_TOKEN": BOT_TOKEN.strip(), "TELEGRAM_API": TELEGRAM_API.strip(),
        "TELEGRAM_HASH": TELEGRAM_HASH.strip(), "OWNER_ID": OWNER_ID.strip(),
        "DATABASE_URL": DATABASE_URL.strip(),
    })

for pair in filter(None, (p.strip() for p in Extra_Config.split(";"))):
    key, sep, value = pair.partition("=")
    if not sep or not re.fullmatch(r"[A-Z][A-Z0-9_]*", key.strip()):
        raise ValueError(f"⚠️ Extra_Config: bad entry {pair!r}, use KEY=VALUE")
    config[key.strip()] = value.strip()

config["BASE_URL"] = f"https://{App_Name}.herokuapp.com"   # the web dyno needs a port, this starts it
config["UPSTREAM_REPO"] = Upstream_Repo.strip()
config["UPSTREAM_BRANCH"] = Upstream_Branch.strip()
config.setdefault("SET_COMMANDS", "True")

problems = []
if not re.fullmatch(r"\d+:[\w-]{20,}", config.get("BOT_TOKEN", "")):
    problems.append("BOT_TOKEN looks wrong")
if not config.get("OWNER_ID", "").isdigit():
    problems.append("OWNER_ID must be digits")
if not config.get("TELEGRAM_API", "").isdigit():
    problems.append("TELEGRAM_API must be digits")
if not re.fullmatch(r"[0-9a-fA-F]{32}", config.get("TELEGRAM_HASH", "")):
    problems.append("TELEGRAM_HASH must be 32 hex characters")
if not config.get("DATABASE_URL", "").startswith(("mongodb://", "mongodb+srv://")):
    problems.append("DATABASE_URL must start with mongodb:// or mongodb+srv://")
if problems:
    raise ValueError("⚠️ " + "; ".join(problems))

# the output of config:set lists the values, so it is not printed
run(["heroku", "config:set", "-a", App_Name, *[f"{k}={v}" for k, v in config.items()]], show=False)
print(f"✅ Saved {len(config)} Config Vars:", ", ".join(config))


In [ ]:
#@title <center><h3>🚀 <b>4 · Deploy</b></h3></center>
workdir = "/content/Leech-TJ-deploy"
if os.path.isdir(workdir):
    shutil.rmtree(workdir)
# a full clone: Heroku refuses a push from a shallow one
run(["git", "clone", "-b", Upstream_Branch.strip(), Upstream_Repo.strip(), workdir])
run(["heroku", "git:remote", "-a", App_Name], cwd=workdir)

print("📦 Pushing, Heroku builds the image now (several minutes)...")
push = subprocess.run(["git", "push", "heroku", "HEAD:main", "-f"], cwd=workdir)
if push.returncode != 0:
    raise RuntimeError("⚠️ The push failed, read the output above")

run(["heroku", "ps:scale", "web=1", "-a", App_Name])
print("✅ Done! Check the logs (cell 5), then send /start to your bot.")


In [ ]:
#@title <center><h3>📜 <b>5 · Logs / Restart / Logout</b></h3></center>
Action = "Show logs"  #@param ["Show logs", "Restart the bot", "Logout (when you are done)"]

if Action == "Show logs":
    print(run(["heroku", "logs", "-n", "200", "-a", App_Name], show=False, check=False)[-6000:])
elif Action == "Restart the bot":
    run(["heroku", "dyno:restart", "-a", App_Name])
else:
    run(["heroku", "logout"], check=False)
    if os.path.exists(os.path.expanduser("~/.netrc")):
        os.remove(os.path.expanduser("~/.netrc"))
    print("🔓 Logged out, the credentials were removed from this session.")
